# Parameter Optimization and Statistical analysis

In [2]:
from bose_harmonic import (
  wavefunction,
  wavefunction_derivative,
  local_energy,
  drift_force,
  HarmonicParams,
)
from vmc import Metropolis
from stats import timeseries_bootstrap
import numpy as np

cycles = 10_000
time_step = 0.05
diffusion_coefficient = 0.5
learning_rate = 0.01
optimization_iterations = 100

number_particles = 100
dim = 3

simulation = Metropolis[HarmonicParams](number_particles, dim)

parameter_guess = HarmonicParams(alpha=0.7)

energy, parameters = simulation.optimize(
  wavefunction,
  wavefunction_derivative,
  local_energy,
  drift_force,
  parameter_guess,
  time_step,
  diffusion_coefficient,
  cycles,
  optimization_iterations,
  learning_rate,
)


cycles_sample = 2**20
_, _, energies = simulation.sample_importance(
  wavefunction,
  local_energy,
  drift_force,
  parameters,
  time_step,
  diffusion_coefficient,
  cycles_sample,
)

print(parameters)

samples = 2**12
block_size = 2**10

result = timeseries_bootstrap(energies, np.mean, samples, block_size)
print(result)

iteration 10/100: E=153.03, grad=5.167e+01, HarmonicParams(alpha=np.float64(0.5999999999999999))
iteration 20/100: E=150.03, grad=6.069e+00, HarmonicParams(alpha=np.float64(0.4999999999999998))
Finished at 20 iterations
Energy=150.00, grad=8.504e-11, HarmonicParams(alpha=np.float64(0.4999999999999998))
HarmonicParams(alpha=np.float64(0.4999999999999998))
BootstrapResult(statistic=150.0, bias=0.0, standard_error=0.0)


In [1]:
import jax.numpy as jnp
import numpy as np
import optax

from bose_harmonic_jax import (
  log_wavefunction_jax,
  wavefunction_derivative_jax,
  local_energy_jax,
  drift_force_jax,
  HarmonicParams,
)
from vmc_jax import MetropolisJAX
from stats import timeseries_bootstrap

cycles = 10_000
time_step = 0.01
diffusion_coefficient = 0.5
optimization_iterations = 200

number_particles = 100
dim = 3

# Optimize parameters
simulation = MetropolisJAX[HarmonicParams](number_particles, dim)

parameter_guess = HarmonicParams(alpha=jnp.array(0.7))

lr_schedue = optax.exponential_decay(
  init_value=0.01,
  transition_steps=optimization_iterations // 2,
  decay_rate=0.5
)
optimizer = optax.adam(learning_rate=lr_schedue)

energy, parameters = simulation.optimize(
  log_wavefunction_jax,
  wavefunction_derivative_jax,
  local_energy_jax,
  drift_force_jax,
  parameter_guess,
  time_step,
  diffusion_coefficient,
  cycles,
  optimization_iterations,
  optimizer,
)

# Bootstrap analysis
cycles_sample = 2**20

_, _, energies = simulation.sample_importance(
  log_wavefunction_jax,
  local_energy_jax,
  drift_force_jax,
  parameters,
  time_step,
  diffusion_coefficient,
  cycles_sample,
)

print(parameters)

samples = 2**12
block_size = 2**10

result = timeseries_bootstrap(np.array(energies), np.mean, samples, block_size)
print(result)


iteration 0/200: E=184.32, grad=6.009e+01, HarmonicParams(alpha=Array(0.69949996, dtype=float32))
iteration 10/200: E=167.22, grad=3.651e+01, HarmonicParams(alpha=Array(0.67259794, dtype=float32))
iteration 20/200: E=151.93, grad=5.047e+00, HarmonicParams(alpha=Array(0.61940444, dtype=float32))
iteration 30/200: E=146.87, grad=9.330e+00, HarmonicParams(alpha=Array(0.5638796, dtype=float32))
iteration 40/200: E=150.99, grad=3.113e+00, HarmonicParams(alpha=Array(0.5380649, dtype=float32))
iteration 50/200: E=149.77, grad=2.121e+00, HarmonicParams(alpha=Array(0.5223691, dtype=float32))
iteration 60/200: E=149.58, grad=8.379e-01, HarmonicParams(alpha=Array(0.51324, dtype=float32))
iteration 70/200: E=150.23, grad=1.684e+00, HarmonicParams(alpha=Array(0.5084818, dtype=float32))
iteration 80/200: E=150.38, grad=1.125e+00, HarmonicParams(alpha=Array(0.50453925, dtype=float32))
iteration 90/200: E=149.50, grad=1.645e+00, HarmonicParams(alpha=Array(0.5025863, dtype=float32))
iteration 100/200: 